# Teach a small model to reason: Qwen 3.5 0.8B, GRPO with MLX, on a Mac

Ask Qwen 3.5 0.8B a maths word problem and tell it to end with an `Answer:` line, and it skips the working: it
writes the answer line straight away and is usually wrong. We fix that with **GRPO**, the reinforcement-learning
method behind DeepSeek-R1: the model tries every problem several times, and the reward only checks the final
number. Nobody shows it a worked solution. The loop is about 100 lines on **mlx-lm** (`grpo.py`, next to this notebook).

In [ ]:
import sys

from mlx_lm import load

from grpo import MODEL, chat, evaluate, load_gsm8k, reward

## 0. The machine

The same MacBook Pro as every lesson in this series. MLX runs on the Mac's GPU.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

## 1. The data

[`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k) (MIT): grade-school maths word problems. Each answer
is a worked solution that ends in `#### <the number>`. We only use that final number.

In [ ]:
train, test = load_gsm8k("train"), load_gsm8k("test")
print(f"train {len(train):,} problems | test {len(test):,} problems")
print(test.question[0])
print("---")
print(test.answer[0])

In [ ]:
test = test.iloc[:200].reset_index(drop=True)   # the same 200 test problems, before and after
test.gold[:5].tolist()

## 2. The reward

The whole reward: **1 if the reply's `Answer:` line holds the right number, else 0.** It never mentions steps.

In [ ]:
print(reward("She sells 16 - 3 - 4 = 9 eggs a day, 9 * 2 = 18 dollars.\nAnswer: 18", "18"))
print(reward("Answer: 12", "18"))
print(reward("She makes 18 dollars a day.", "18"))   # right number, but no Answer line

## 3. Before: the base model

Qwen 3.5 0.8B (Apache-2.0) in **no-think mode**. The prompt is the problem plus one instruction.

In [ ]:
model, tok = load(MODEL)
print(tok.decode(chat(tok, test.question[0])))

In [ ]:
base = evaluate(model, tok, test, name="base model")
for i in range(3):
    print(f"--- gold {test.gold[i]}:", repr(base.reply[i]))

It writes the `Answer:` line at once, with no working. That's not because it can't do the maths: asked the
question **alone**, with no instruction, it writes its working and lands the right number much more often
(we score its last number here). It just doesn't think before it commits to the answer line.

In [ ]:
alone = evaluate(model, tok, test, ask=False, name="base model, question alone")

In [ ]:
del model
import gc, mlx.core as mx
gc.collect(); mx.clear_cache()

## 4. GRPO

Every step, `grpo.py`:
1. takes **8** new training problems and samples **8** answers to each (temperature 1),
2. scores every answer with the reward (1 or 0),
3. compares each answer with the other 7 for the same problem: better than the group's average → pushed up, worse → pushed down,
4. updates small **LoRA** matrices on every layer (the model's own weights stay frozen).

4 steps = 32 problems, 256 answers.

In [ ]:
!{sys.executable} grpo.py --steps 4 --lr 2e-5 --out runs/grpo

## 5. After

The same 200 problems, the same prompt, greedy decoding.

In [ ]:
model, tok = load(MODEL, adapter_path="runs/grpo")
after = evaluate(model, tok, test, name="after GRPO")

In [ ]:
fixed = test.index[~base.ok & after.ok]
print(f"now right: {len(fixed)} problems | now wrong: {(base.ok & ~after.ok).sum()} problems")
for i in fixed[:3]:
    print(f"\n=== {test.question[i]}\n--- gold {test.gold[i]} | before: {base.reply[i]!r}\n--- after:\n{after.reply[i]}")

In [ ]:
import json
run = json.load(open("runs/grpo/run.json"))
print(f"MacBook Pro M5 Max, 36 GB | Qwen 3.5 0.8B, GRPO with mlx-lm (LoRA) | 4 steps, "
      f"{int(run['train_s'] // 60)} min {run['train_s'] % 60:.0f} s, peak {run['peak_gb']:.1f} GB")
print(f"200 GSM8K test problems: base {100 * base.ok.mean():.1f} %  ->  after GRPO {100 * after.ok.mean():.1f} %")
print(f"median reply: {int(base.tokens.median())} tokens  ->  {int(after.tokens.median())} tokens")

## 6. Run it in Ollama

Fuse the adapter into the model, turn it into a GGUF file with llama.cpp's converter (the copy Unsloth Studio
installed in lesson 4), and keep it at 8 bits (Q8_0). One small fix first: mlx-lm stores two kinds of Qwen 3.5
weights in a slightly different form than the converter expects.

In [ ]:
del model
gc.collect(); mx.clear_cache()
from pathlib import Path

from grpo import fix_for_llama_cpp, modelfile

LLAMA_CPP = Path.home() / ".unsloth/llama.cpp"
STUDIO_PYTHON = Path.home() / ".unsloth/studio/unsloth_studio/bin/python"
!mlx_lm.fuse --model {MODEL} --adapter-path runs/grpo --save-path runs/qwen-reasoner > runs/fuse.log 2>&1
fix_for_llama_cpp("runs/qwen-reasoner", "runs/qwen-reasoner-hf")
!{STUDIO_PYTHON} {LLAMA_CPP}/convert_hf_to_gguf.py runs/qwen-reasoner-hf --no-mtp --outfile runs/reasoner-bf16.gguf --outtype bf16 > runs/convert.log 2>&1; tail -1 runs/convert.log
!{LLAMA_CPP}/llama-quantize runs/reasoner-bf16.gguf runs/reasoner-Q8_0.gguf Q8_0 > runs/quantize.log 2>&1; tail -2 runs/quantize.log
!ls -lh runs/*.gguf

In [ ]:
open("runs/Modelfile", "w").write(modelfile("reasoner-Q8_0.gguf"))
print(open("runs/Modelfile").read())
!cd runs && ollama create reasoner -f Modelfile

In [ ]:
from grpo import evaluate_ollama

ollama_ok = evaluate_ollama("reasoner", test, "after GRPO")

## 7. Put it on Hugging Face

The LoRA adapter (for MLX) and the Q8_0 file with its Modelfile (for Ollama) go to a Hugging Face model repo.
Log in once with `hf auth login`.

In [ ]:
from huggingface_hub import HfApi

REPO = "kgptalkie/qwen3.5-0.8b-grpo-reasoning"
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path="runs/grpo", repo_id=REPO, allow_patterns=["adapters.safetensors", "adapter_config.json"])
api.upload_file(path_or_fileobj="runs/reasoner-Q8_0.gguf", path_in_repo="reasoner-Q8_0.gguf", repo_id=REPO)
api.upload_file(path_or_fileobj="runs/Modelfile", path_in_repo="Modelfile", repo_id=REPO)
api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO)
print(f"https://huggingface.co/{REPO}")